In [4]:
import pandas as pd
import os
import numpy as np
import shutil

# --- Configuration ---
# Define the names of the input and output folders.
# These folders are assumed to be in the same directory as this script.
input_folder_name = 'weeklystats'
output_folder_name = 'EngineeredFeatures'

# Get the path to the current working directory
base_path = os.getcwd()

# Construct the full paths for the input and output folders
input_folder_path = os.path.join(base_path, input_folder_name)
output_folder_path = os.path.join(base_path, output_folder_name)

# --- Main Script ---

# Check if the input folder exists
if not os.path.exists(input_folder_path):
    print(f"Error: The input folder '{input_folder_path}' was not found. Please ensure it exists.")
else:
    # Delete the output folder if it exists, to ensure a clean run.
    if os.path.exists(output_folder_path):
        shutil.rmtree(output_folder_path)
        print(f"Deleted existing output folder: {output_folder_path}")
        
    # Create the new output folder
    os.makedirs(output_folder_path)
    print(f"Created new output folder: {output_folder_path}")

    # Get a list of all CSV files in the input folder
    weekly_files = [f for f in os.listdir(input_folder_path) if f.endswith('.csv') and not f.startswith('.')]

    if not weekly_files:
        print(f"No CSV files found in the '{input_folder_name}' folder. Nothing to process.")
    else:
        # Columns to be dropped from the raw data
        columns_to_drop = ['Series', 'No. of Trades', 'Turnover ₹', 'Deliverable Qty', 'Last Price', 'Average Price']

        # Loop through each weekly file
        for file_name in weekly_files:
            print(f"\n--- Processing file: {file_name} ---")
            
            # Construct the full path to the current file
            file_path = os.path.join(input_folder_path, file_name)

            try:
                # Read the combined daily data into a pandas DataFrame.
                df = pd.read_csv(file_path, thousands=',')
                
                # Ensure the 'Date' column is in datetime format and set it as the index
                df['Date'] = pd.to_datetime(df['Date'])
                df.set_index('Date', inplace=True)

                # Drop the initial unnecessary columns
                df = df.drop(columns=columns_to_drop, errors='ignore')

                # --- Intermediate Calculations for Normalization ---
                # These are temporary and will be dropped later
                df['20_SMA'] = df['Close Price'].rolling(window=20).mean()
                df['20_STD'] = df['Close Price'].rolling(window=20).std()
                df['VWAP'] = (df['Close Price'] * df['Total Traded Quantity']).cumsum() / df['Total Traded Quantity'].cumsum()
                
                # --- Price-Agnostic Feature Engineering ---
                
                # --- Price Movement Features (Normalized) ---
                # Weekly Return as a percentage change
                df['Weekly_Return'] = df['Close Price'].pct_change() * 100
                
                # Normalize Open, High, and Low prices against the Close Price
                df['Normalized_Open'] = df['Open Price'] / df['Close Price']
                df['Normalized_High'] = df['High Price'] / df['Close Price']
                df['Normalized_Low'] = df['Low Price'] / df['Close Price']
                
                # Normalize EMAs as a percentage difference from the Close Price.
                ema5 = df['Close Price'].ewm(span=5, adjust=False).mean()
                ema13 = df['Close Price'].ewm(span=13, adjust=False).mean()
                ema23 = df['Close Price'].ewm(span=23, adjust=False).mean()
                df['EMA5_Diff_Pct'] = (df['Close Price'] - ema5) / df['Close Price'] * 100
                df['EMA13_Diff_Pct'] = (df['Close Price'] - ema13) / df['Close Price'] * 100
                df['EMA23_Diff_Pct'] = (df['Close Price'] - ema23) / df['Close Price'] * 100

                # --- Volume-Based Features ---
                # Volume SMA over 20 weeks
                df['Volume_SMA20'] = df['Total Traded Quantity'].rolling(window=20).mean()
                df['Relative_Volume'] = df['Total Traded Quantity'] / df['Volume_SMA20']
                
                # Volume Spike Ratio (5-week average)
                df['Volume_SMA5'] = df['Total Traded Quantity'].rolling(window=5).mean()
                df['Volume_Spike_Ratio_5Wk'] = df['Total Traded Quantity'] / df['Volume_SMA5']
                
                # On-Balance Volume (OBV)
                # A running total of volume based on price direction
                obv_series = pd.Series(0, index=df.index)
                obv_series.iloc[0] = df['Total Traded Quantity'].iloc[0]
                for i in range(1, len(df)):
                    if df['Close Price'].iloc[i] > df['Close Price'].iloc[i-1]:
                        obv_series.iloc[i] = obv_series.iloc[i-1] + df['Total Traded Quantity'].iloc[i]
                    elif df['Close Price'].iloc[i] < df['Close Price'].iloc[i-1]:
                        obv_series.iloc[i] = obv_series.iloc[i-1] - df['Total Traded Quantity'].iloc[i]
                    else:
                        obv_series.iloc[i] = obv_series.iloc[i-1]
                df['OBV'] = obv_series
                # To make OBV price-agnostic, we'll normalize it by its 20-week SMA
                df['Normalized_OBV'] = df['OBV'] / df['OBV'].rolling(window=20).mean()
                
                # --- Volatility-Based Features ---
                # Normalized Weekly Range (Range as a percentage of the Close Price)
                df['Normalized_Weekly_Range'] = (df['High Price'] - df['Low Price']) / df['Close Price'] * 100
                
                # Average True Range (ATR) over a 14-week period
                df['High-Low'] = df['High Price'] - df['Low Price']
                df['High-PrevClose'] = abs(df['High Price'] - df['Close Price'].shift(1))
                df['Low-PrevClose'] = abs(df['Low Price'] - df['Close Price'].shift(1))
                df['True_Range'] = df[['High-Low', 'High-PrevClose', 'Low-PrevClose']].max(axis=1)
                df['ATR'] = df['True_Range'].ewm(span=14, adjust=False).mean()
                
                # Bollinger Band Z-Score based on 20-week SMA
                df['Bollinger_Z_Score'] = (df['Close Price'] - df['20_SMA']) / df['20_STD']
                
                # --- Momentum-Based Features ---
                # Rate of Change (ROC) over 10 weeks
                df['ROC_10'] = df['Close Price'].pct_change(periods=10) * 100
                
                # Relative Strength Index (RSI) over a 14-week period
                delta = df['Close Price'].diff(1)
                gain = delta.where(delta > 0, 0)
                loss = -delta.where(delta < 0, 0)
                avg_gain = gain.ewm(span=14, adjust=False).mean()
                avg_loss = loss.ewm(span=14, adjust=False).mean()
                rs = avg_gain / avg_loss
                df['RSI'] = 100 - (100 / (1 + rs))

                # Stochastic Oscillator (%K and %D)
                lowest_low = df['Low Price'].rolling(window=14).min()
                highest_high = df['High Price'].rolling(window=14).max()
                df['Stochastic_%K'] = ((df['Close Price'] - lowest_low) / (highest_high - lowest_low)) * 100
                df['Stochastic_%D'] = df['Stochastic_%K'].rolling(window=3).mean()
                
                # VWAP Difference as a percentage
                df['VWAP_Diff_Pct'] = (df['Close Price'] - df['VWAP']) / df['VWAP'] * 100

                # --- Final Cleanup: Drop all raw price columns and temporary calculations ---
                df = df.drop(columns=['Open Price', 'High Price', 'Low Price', 'Close Price', 
                                      '20_SMA', '20_STD', 'VWAP', 'OBV', 'Volume_SMA5', 
                                      'High-Low', 'High-PrevClose', 'Low-PrevClose', 'True_Range'], errors='ignore')
                
                # Reset the index to make 'Date' a column again before saving
                df.reset_index(inplace=True)

                # Create a new output file name and path
                output_file_path = os.path.join(output_folder_path, file_name.replace('-Combined-Weekly-Data.csv', '-Engineered-Features.csv'))
                
                # Save the new weekly data to a CSV file
                df.to_csv(output_file_path, index=False)
                
                print(f"Successfully engineered features for '{file_name}' and saved as '{os.path.basename(output_file_path)}'.")
                print(f"Total rows in new file: {len(df)}")
                
            except Exception as e:
                print(f"An error occurred while processing {file_name}: {e}")

print("\n--- All files have been processed ---")


Deleted existing output folder: /Users/niranjansmacbook/PycharmProjects/DBA/DBA-Capstone/Stocks/RawData/EngineeredFeatures
Created new output folder: /Users/niranjansmacbook/PycharmProjects/DBA/DBA-Capstone/Stocks/RawData/EngineeredFeatures

--- Processing file: ICICI-Combined-Weekly-Data.csv ---
Successfully engineered features for 'ICICI-Combined-Weekly-Data.csv' and saved as 'ICICI-Engineered-Features.csv'.
Total rows in new file: 575

--- Processing file: Tata-Combined-Weekly-Data.csv ---
Successfully engineered features for 'Tata-Combined-Weekly-Data.csv' and saved as 'Tata-Engineered-Features.csv'.
Total rows in new file: 575

--- Processing file: SBI-Combined-Weekly-Data.csv ---
Successfully engineered features for 'SBI-Combined-Weekly-Data.csv' and saved as 'SBI-Engineered-Features.csv'.
Total rows in new file: 575

--- Processing file: Bajaj-Combined-Weekly-Data.csv ---
Successfully engineered features for 'Bajaj-Combined-Weekly-Data.csv' and saved as 'Bajaj-Engineered-Feature